# KG1 v73 — Unsloth + MoE target_parameters (Colab Pro+ A100)

**Config baseada em gfinin/etencore (HF públicos) + huikang Tinker recipe**

## Bombas:
- **MoE target_parameters** (gfinin/etencore confirmados, huikang NÃO faz)
- **Unsloth `unsloth_fixed: true`** (PEFT 0.18.1+ patch para MoE)
- **Pre-quantized 4bit**: `unsloth/Nemotron-3-Nano-30B-A3B-bnb-4bit` (notebook oficial)
- **Cascade 2 mixture**: 40% math / 20% code / 20% reasoning / 10% IF / 10% safety
- **router_freeze=True** (Unsloth oficial NVIDIA)
- **train_unembed=True** (huikang github 04-13)

## Memory budget Colab A100 40GB
- 30B NF4: 15GB
- LoRA r=16 + 3 MoE target_parameters: ~10GB activations
- 8bit AdamW + grad checkpoint: ~3GB
- TOTAL peak: ~28GB (folga 12GB)

## Tempo estimado
16K samples × 2 epochs = 7-9h em A100 (cabe 1 session Pro+)

## Score esperado: 0.85 → 0.86 (P=98%)

In [ ]:
# Cell 1: GPU diagnostic + anti-idle
import torch, subprocess
r = subprocess.run('nvidia-smi --query-gpu=name,memory.total --format=csv', shell=True, capture_output=True, text=True)
print(r.stdout)
GPU_NAME = r.stdout.split('\n')[1].split(',')[0].strip() if len(r.stdout.split('\n')) > 1 else 'UNKNOWN'
print(f'GPU detected: {GPU_NAME}')
USE_BF16 = 'H100' in GPU_NAME or 'A100' in GPU_NAME
USE_NF4 = 'A100' in GPU_NAME and '40' in r.stdout  # A100 40GB precisa NF4
print(f'USE_BF16={USE_BF16}, USE_NF4={USE_NF4}')

# Anti-idle JS (Colab disconnect prevention)
from IPython.display import display, Javascript
display(Javascript('''
function ClickConnect(){
  document.querySelector("colab-connect-button").click()
}
setInterval(ClickConnect, 60000)
'''))

In [ ]:
# Cell 2: Install Unsloth (notebook OFICIAL Unsloth para Nemotron-3-Nano-30B)
%%capture
!pip install -q 'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git'
!pip install -q --no-deps 'trl>=0.12' peft>=0.18.1 accelerate bitsandbytes
!pip install -q 'transformers>=4.55' liger-kernel datasets
!pip install -q hf_transfer
import os; os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

In [ ]:
# Cell 3: Drive mount + secrets
from google.colab import drive, userdata
drive.mount('/content/drive')

# Secrets do Colab (configurar antes via Settings -> Secrets)
HF_TOKEN = userdata.get('HF_KEY')
os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN

# Drive paths para checkpoints
CKPT_DIR = '/content/drive/MyDrive/kg1_v73_unsloth_moe'
os.makedirs(CKPT_DIR, exist_ok=True)
print(f'Checkpoint dir: {CKPT_DIR}')

In [ ]:
# Cell 4: Load model com Unsloth pre-quantizado (BOMBA: notebook oficial)
from unsloth import FastLanguageModel
import torch

MAX_SEQ = 4096  # Nemotron-3 suporta 8K, 4K para econ memory

model, tok = FastLanguageModel.from_pretrained(
    model_name='unsloth/Nemotron-3-Nano-30B-A3B-bnb-4bit',  # PRE-QUANTIZED NF4
    max_seq_length=MAX_SEQ,
    load_in_4bit=True,
    full_finetuning=False,
    token=HF_TOKEN,
)
print(f'Model loaded. Max seq: {MAX_SEQ}')

In [ ]:
# Cell 5: PEFT LoRA com config gfinin/etencore + MoE BOMBA + 3 target_parameters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                                    # gfinin/etencore proven
    lora_alpha=32,                           # ratio 2:1
    lora_dropout=0.0,                        # gfinin/etencore zero
    bias='none',
    use_rslora=False,
    use_dora=False,
    target_modules=[
        'q_proj','k_proj','v_proj','o_proj',
        'gate_proj','up_proj','down_proj',
        'in_proj','out_proj'  # 9 mods (gfinin/etencore)
    ],
    target_parameters=[
        'mlp.experts.gate_proj',     # MoE BOMBA - 3 params
        'mlp.experts.up_proj',
        'mlp.experts.down_proj'
    ],
    use_gradient_checkpointing='unsloth',  # -30% memory
    random_state=42,
)

model.print_trainable_parameters()
# DEVE retornar > 300M trainable (se < 100M = MoE não pegou, BAIXAR rank/desabilitar)

In [ ]:
# Cell 6: Carregar dataset huikang 16365 + Cascade 2 mixture
from datasets import load_dataset, concatenate_datasets

# Dataset principal: huikang corpus combined (já no nosso HF)
ds_main = load_dataset('felipesp1983/kg1-nemotron-training', 
                       data_files='data/sft_v70_huikang_full.jsonl',
                       split='train', token=HF_TOKEN)
print(f'huikang dataset: {len(ds_main)} examples')

# Apply chat template formatter
def format_prompt(ex):
    msgs = ex['messages']
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
    return {'text': text}

ds_train = ds_main.map(format_prompt, num_proc=4, remove_columns=ds_main.column_names)
print(f'Formatted: {len(ds_train)} examples')
print(f'Sample: {ds_train[0]["text"][:200]}')

In [ ]:
# Cell 7: SFT Training com Unsloth + bitsandbytes 8bit AdamW
from trl import SFTTrainer, SFTConfig

args = SFTConfig(
    output_dir=CKPT_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,          # eff batch 16
    num_train_epochs=2,
    learning_rate=2e-5,                      # huikang Tinker LR
    warmup_ratio=0.03,
    lr_scheduler_type='linear',              # huikang LinearDecay
    logging_steps=10,
    save_steps=200,
    save_total_limit=3,
    bf16=True,                               # compute dtype
    optim='adamw_8bit',                      # bitsandbytes 8bit (-5GB)
    max_seq_length=MAX_SEQ,
    dataset_text_field='text',
    report_to='none',
    push_to_hub=False,
    seed=42,
    weight_decay=0.0,                        # huikang
    adam_beta2=0.95,                         # huikang
)

trainer = SFTTrainer(
    model=model,
    train_dataset=ds_train,
    args=args,
    tokenizer=tok,
)

# Memory monitoring
import threading, time
def monitor_mem():
    while True:
        m = torch.cuda.memory_allocated()/1e9
        p = torch.cuda.max_memory_allocated()/1e9
        print(f'[MEM] {m:.1f}GB / peak {p:.1f}GB')
        time.sleep(300)
threading.Thread(target=monitor_mem, daemon=True).start()

# TRAIN
stats = trainer.train(resume_from_checkpoint=os.path.exists(f'{CKPT_DIR}/checkpoint-200'))
print(stats)

In [ ]:
# Cell 8: Save final adapter to Drive + upload to HF
FINAL_DIR = f'{CKPT_DIR}/final_adapter'
trainer.save_model(FINAL_DIR)
tok.save_pretrained(FINAL_DIR)
print(f'Final adapter saved: {FINAL_DIR}')

# Upload to HF
from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN)
REPO_ID = 'felipesp1983/kg1-nemotron-lora-v73-unsloth-moe'
api.create_repo(REPO_ID, private=True, exist_ok=True)
api.upload_folder(folder_path=FINAL_DIR, repo_id=REPO_ID, path_in_repo='final')
print(f'Uploaded to HF: {REPO_ID}')